In [1]:
import numpy as np
import torch
import time
from transformers import Trainer, TrainingArguments

from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from evaluation.metrics_trainer_callback import SaveMetricsCallback
from evaluation.lr_scheduler_callback import ReduceLROnPlateauCallback

from evaluation.metrics import compute_ece, compute_B_std, compute_B_mean
from data_loading.get_datasets import get_glue_dataset
from models.get_roberta import get_hypernet_on_last_layer_roberta

/home/lewy700/Documents/roberta-hypernet-custom/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

glue_dataset_name = "cola"
model_name = "roberta-base"
lora_r = 1
lora_alpha = 16
hypernet_hidden_dim = 16

print(f"Hypernet on: {glue_dataset_name}")

Hypernet on: cola


In [3]:
model, tokenizer, hypernet = get_hypernet_on_last_layer_roberta(model_name=model_name, lora_r=lora_r, lora_alpha=lora_alpha, hypernet_hidden_dim=hypernet_hidden_dim, use_on_value_matrix=False)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
encoded_dataset, metric = get_glue_dataset(glue_dataset_name, tokenizer, truncation=True, max_length=512)

In [5]:
for name, param in model.named_parameters():
        if "hypernet" in name:
            print(name, param.requires_grad)

base_model.model.roberta.encoder.layer.11.attention.self.query.lora_A.default.hypernet.fc1.weight True
base_model.model.roberta.encoder.layer.11.attention.self.query.lora_A.default.hypernet.fc1.bias True
base_model.model.roberta.encoder.layer.11.attention.self.query.lora_A.default.hypernet.fc2.weight True
base_model.model.roberta.encoder.layer.11.attention.self.query.lora_A.default.hypernet.fc2.bias True
base_model.model.roberta.encoder.layer.11.attention.self.query.lora_A.default.hypernet.embedding.weight True


In [6]:
def compute_metrics(eval_pred):
        logits, labels = eval_pred
        probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()

        predictions = np.argmax(probs, axis=-1)
        results = metric.compute(predictions=predictions, references=labels)

        results["ece"] = compute_ece(probs, labels)
        results["hyper_B_std"] = compute_B_std(hypernet, device=device)
        results["hyper_B_mean"] = compute_B_mean(hypernet, device=device)

        return results

In [7]:
training_args = TrainingArguments(
    output_dir=f"./outputs/hypernet_{glue_dataset_name}",
    eval_strategy="steps",
    eval_steps=40,
    save_strategy="steps",
    save_steps=1000000000,
    learning_rate=4e-4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=32,
    num_train_epochs=20, # 80
    metric_for_best_model="matthews_correlation",
    dataloader_num_workers=4,
    warmup_ratio=0.0,
    lr_scheduler_type="constant",
    optim="adamw_torch",
    weight_decay=0.1,
    disable_tqdm=False
)

optimizer = AdamW(model.parameters(), lr=0.1, weight_decay=0.1)

scheduler = ReduceLROnPlateau(
    optimizer,
    mode="min",
    patience=5
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[
        ReduceLROnPlateauCallback(),
        SaveMetricsCallback(f"./results", f"hypernet_{glue_dataset_name}_{str(int(time.time()))}.csv")
    ],
    optimizers=(optimizer, scheduler)
)

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [8]:
# def grad_hook_h(grad):
#     print(f"[BACKWARD] grad shape: {grad.shape}, mean: {grad.abs().mean().item():.8f} HYPERNET")
#     return grad

# def grad_hook(grad):
#     print(f"[BACKWARD] grad shape: {grad.shape}, mean: {grad.abs().mean().item():.8f}")
#     return grad

# hypernet.fc1.weight.register_hook(grad_hook_h)
# model.roberta.encoder.layer[-2].attention.self.query.lora_A["default"].weight.register_hook(grad_hook)

In [9]:
trainer.train()

Step,Training Loss,Validation Loss,Matthews Correlation,Ece,Hyper B Std,Hyper B Mean
40,No log,17.119318,0.000000,0.691275,7.839802,-0.188033
80,No log,4.230561,0.000000,0.308724,7.388470,-0.125842
120,No log,7.426535,0.000000,0.308725,2.608974,0.011976
160,No log,7.837355,0.000000,0.691263,1.595004,0.005119


KeyboardInterrupt: 

In [1]:
from datasets import load_dataset
import numpy as np
import evaluate
import torch
from torch import nn
from transformers import RobertaTokenizer, RobertaModel, RobertaForSequenceClassification, Trainer, TrainingArguments
from transformers.modeling_outputs import SequenceClassifierOutput
from peft import get_peft_model, LoraConfig, TaskType, PeftModel, PeftConfig

from evaluation.metrics import compute_ece, compute_B_std

print("Hypernet CoLA")

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "roberta-base"

for i in range(2):
    print(f"=== Run {i} ==============")

    tokenizer = RobertaTokenizer.from_pretrained(model_name)

    def tokenize_function(examples):
        return tokenizer(examples["sentence"], truncation=True, padding="max_length", max_length=512)

    dataset = load_dataset("glue", "cola")

    encoded_dataset = dataset.map(tokenize_function, batched=True)
    encoded_dataset = encoded_dataset.rename_column("label", "labels")
    encoded_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

    metric = evaluate.load("glue", "cola")

    class LoRAHyperNet(nn.Module):
        def __init__(self, input_dim, hidden_dim, lora_dim):
            super().__init__()

            self.lora_r = lora_dim
            self.hidden_dim = hidden_dim
            self.input_dim = input_dim

            self.fc1 = nn.Linear(lora_dim * input_dim, hidden_dim)
            self.fc2 = nn.Linear(hidden_dim, input_dim * lora_dim)

        def forward(self, inpt, layer_id=0):
            flat = inpt.view(-1)
            h = torch.relu(self.fc1(flat))
            opt = self.fc2(h).view(inpt.shape[1], inpt.shape[0])  # B shape: [in_dim, r]

            return opt

    class DynamicLoRALayer(nn.Module):
        def __init__(self, hidden_size, r, hypernet: nn.Module):
            super().__init__()
            self.hidden_size = hidden_size
            self.r = r
            self.hypernet = hypernet

            self.weight = torch.tensor(0.0, dtype=self.hypernet.fc1.weight.dtype)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            if self.training:
                A = torch.randn((self.r, self.hidden_size)).to(x.device)

                B = self.hypernet(A)  # A: [r, hidden], B: [hidden, r]
                output = torch.matmul(torch.matmul(x, B), A)  # returns [batch, seq_len, hidden]
            else:
                with torch.no_grad():
                    A = torch.randn((self.r, self.hidden_size)).to(x.device)

                    B = self.hypernet(A)  # A: [r, hidden], B: [hidden, r]
                    output = torch.matmul(torch.matmul(x, B), A)  # returns [batch, seq_len, hidden]
            return output


    base_model = RobertaForSequenceClassification.from_pretrained(model_name)

    lora_r = 1
    base_hidden_size = 768
    hypernet_hidden_dim = 16

    hypernet = LoRAHyperNet(base_hidden_size, hypernet_hidden_dim, lora_r)

    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        inference_mode=False,
        r=lora_r,
        lora_alpha=16,
    )

    def forward_hook(module, input, output):
        if hasattr(module, 'weight'):
            print(f"[WEIGHTS] {module.__class__.__name__} weight mean: {module.weight.data[0][:10]}")
        
    def grad_hook(grad):
        print(f"[BACKWARD] grad shape: {grad.shape}, mean: {grad.mean().item():.8f}")
        return grad
        
    def grad_hook_h(grad):
        print(f"[BACKWARD] grad shape: {grad.shape}, mean: {grad.mean().item():.8f} HYPERNET")
        return grad

    base_model = get_peft_model(base_model, peft_config=peft_config)

    # target_layer = hypernet.fc1  # Example layer
    # hook_handle = target_layer.register_forward_hook(forward_hook)
    # target_layer.weight.register_hook(grad_hook_h)
    # base_model.roberta.encoder.layer[-2].attention.self.query.lora_A["default"].weight.register_hook(grad_hook)
    # base_model.roberta.encoder.layer[-3].attention.self.query.lora_A["default"].weight.register_hook(grad_hook)

    dynamic_lora_layer = DynamicLoRALayer(base_hidden_size, lora_r, hypernet)

    adapter_name = "default"

    base_model.roberta.encoder.layer[-1].attention.self.query.lora_A[adapter_name] = dynamic_lora_layer
    base_model.roberta.encoder.layer[-1].attention.self.query.lora_B[adapter_name] = nn.Identity()

    total_params = sum(p.numel() for p in base_model.parameters())
    trainable_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    for name, param in base_model.named_parameters():
        if "hypernet" in name:
            print(name, param.requires_grad)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()

        predictions = np.argmax(probs, axis=-1)
        results = metric.compute(predictions=predictions, references=labels)

        # Compute ECE
        ece = compute_ece(probs, labels)
        results["ece"] = ece

        # Compute B std from hypernet (only during eval)
        b_std = compute_B_std(hypernet, device=device)
        results["hyper_B_std"] = b_std

        return results


    training_args = TrainingArguments(
        output_dir="./outputs/hypernet_cola",
        eval_strategy="epoch",
        save_strategy="steps",
        save_steps=1000000000,
        learning_rate=4e-4,
        per_device_train_batch_size=16, # 16
        gradient_accumulation_steps=2, # 2
        per_device_eval_batch_size=32,
        num_train_epochs=80, # 80
        logging_dir="./logs/hypernet_cola",
        logging_strategy="epoch",
        # logging_steps=10,
        metric_for_best_model="matthews_correlation",
        dataloader_num_workers=4,
        warmup_ratio=0.06,
        lr_scheduler_type="linear",
        optim="adamw_torch",
        weight_decay=0.1,
        disable_tqdm=False
    )

    trainer = Trainer(
        model=base_model,
        args=training_args,
        train_dataset=encoded_dataset["train"],
        eval_dataset=encoded_dataset["validation"],
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()

/home/lewy700/Documents/roberta-hypernet-custom/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hypernet CoLA
=== Run 0 ==============


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total parameters: 125,299,988
Trainable parameters: 652,818
base_model.model.roberta.encoder.layer.11.attention.self.query.lora_A.default.hypernet.fc1.weight True
base_model.model.roberta.encoder.layer.11.attention.self.query.lora_A.default.hypernet.fc1.bias True
base_model.model.roberta.encoder.layer.11.attention.self.query.lora_A.default.hypernet.fc2.weight True
base_model.model.roberta.encoder.layer.11.attention.self.query.lora_A.default.hypernet.fc2.bias True


No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Epoch,Training Loss,Validation Loss,Matthews Correlation,Ece,Hyper B Std
1,0.620400,0.587571,0.000000,0.073995,0.147102
2,0.540900,0.529250,0.331308,0.083734,0.154791
3,0.490100,0.472776,0.500624,0.049376,0.150673


KeyboardInterrupt: 

In [2]:
import time

import numpy as np
import torch
from transformers import Trainer, TrainingArguments

from data_loading.get_datasets import get_glue_dataset
from evaluation.metrics import compute_B_mean, compute_B_std, compute_ece
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from evaluation.metrics_trainer_callback import SaveMetricsCallback
from evaluation.lr_scheduler_callback import ReduceLROnPlateauCallback
from models.get_roberta import get_hypernet_on_last_layer_roberta

device = "cuda" if torch.cuda.is_available() else "cpu"

glue_dataset_name = "cola"
model_name = "roberta-base"
lora_r = 1
lora_alpha = 16
hypernet_hidden_dim = 128
hypernet_embeddings_dim = 128

print(f"Hypernet on: {glue_dataset_name}")

for i in range(1):
    print(f"=== Run {i} ==============")

    model, tokenizer, hypernet = get_hypernet_on_last_layer_roberta(model_name=model_name, lora_r=lora_r, lora_alpha=lora_alpha, hypernet_hidden_dim=hypernet_hidden_dim, hypernet_embeddings_dim=hypernet_embeddings_dim, use_on_value_matrix=False, hypernet_with_embedding_input=True)
    encoded_dataset, metric = get_glue_dataset(glue_dataset_name, tokenizer, truncation=True, max_length=512)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()

        predictions = np.argmax(probs, axis=-1)
        results = metric.compute(predictions=predictions, references=labels)

        results["ece"] = compute_ece(probs, labels)
        results["hyper_B_std"] = compute_B_std(hypernet, device=device)
        results["hyper_B_mean"] = compute_B_mean(hypernet, device=device)

        return results

    training_args = TrainingArguments(
        output_dir=f"./outputs/hypernet_{glue_dataset_name}",
        eval_strategy="epoch",
        # eval_steps=4,
        save_strategy="steps",
        save_steps=1000000000,
        learning_rate=4e-4,
        per_device_train_batch_size=16,
        gradient_accumulation_steps=2,
        per_device_eval_batch_size=32,
        num_train_epochs=20, # 80
        logging_dir=f"./logs/hypernet_{glue_dataset_name}",
        logging_strategy="epoch",
        # logging_steps=10,
        metric_for_best_model="matthews_correlation",
        dataloader_num_workers=4,
        warmup_ratio=0.06,
        lr_scheduler_type="linear",
        optim="adamw_torch",
        weight_decay=0.1,
        disable_tqdm=False
    )

    optimizer = AdamW(model.parameters(), lr=0.1, weight_decay=0.1)

    scheduler = ReduceLROnPlateau(
        optimizer,
        mode="min",
        patience=5
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=encoded_dataset["train"],
        eval_dataset=encoded_dataset["validation"],
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[
            # ReduceLROnPlateauCallback(),
            SaveMetricsCallback(f"./results", f"query_hypernet_{glue_dataset_name}_{str(int(time.time()))}.csv")
        ],
        # optimizers=(optimizer, scheduler)
    )

    # print("Parameters that require gradients:")
    # for name, param in model.named_parameters():
    #     if param.requires_grad:
    #         print(name)


    trainer.train()




Hypernet on: cola
=== Run 0 ==============


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
